# Output data integrity checks: tas, tasmax, tasmin

From issue [450](https://github.com/carbonplan/srm-downscaling/issues/450)

```
there are no nans in the output
no time slices are exactly the same
output variables are within broadly reasonable ranges (same as the input data checks in Add input data checks #316)
```

Four checks against the consolidated output store, one section each:


1. **No NaNs**
2. **No duplicate time slices**
3. **Reasonable ranges** (`VAR_SPATIAL_RANGES` in `srm.qaqc`, same bounds as the input checks in #316)
4. **Within input time bounds** (`resolve_member_time_bounds`)

#### Note: Ran on a m8g.4xlarge, but not a hard req
Also, used experimental scheduler, frisky. 

`uv add -U 'frisky>=0.3.0' 'dask-array>=0.3.0' 'git+https://github.com/dask/dask@main' 'git+https://github.com/pydata/xarray@main'`

In [1]:
import os

import dask
import icechunk
import numpy as np
import pandas as pd
import xarray as xr
import zarr

from srm.config import _icechunk_storage_for_path
from srm.qaqc import VAR_SPATIAL_RANGES
from srm.validation import resolve_member_time_bounds

os.environ["FRISKY_SUMMARY"] = "off"
from dask_array.xarray import register
from distributed import Client
from frisky import hijack

zarr.config.set({"async.concurrency": 128})

In [2]:
register()
client = hijack(Client(n_workers=16))
client

<frisky.Client: scheduler="127.0.0.1:45115" id="client-0">

In [3]:
STORE_URI = "s3://carbonplan-srm/output/production/CESM2-WACCM-ERA5-global.icechunk"

repo = icechunk.Repository.open(_icechunk_storage_for_path(STORE_URI))

session = repo.readonly_session("v0.10.0")

tree = xr.open_datatree(session.store, engine="zarr", chunks="auto")
tree

,Array,Chunk
Bytes,98.89 GiB,117.04 MiB
Shape,"(25568, 721, 1440)","(25568, 25, 48)"
Nodes,1,
,Array,Chunk
Bytes,98.89 GiB,117.04 MiB
Shape,"(25568, 721, 1440)","(25568, 25, 48)"
Nodes,1,


In [4]:
GCM = "CESM2-WACCM"
VARIABLES = ["tas", "tasmax"]  # "tasmin"]
SPATIAL = ["lat", "lon"]

# Top-level scenario groups, plus their bias-corrected counterparts nested under
# debiased_coarse/{scenario} -- same var/member layout, different subtree path.
SCENARIO_LABELS = {"historical": "historical", "ssp245": "SSP245", "g6_1p5k": "G6-1.5K"}
GROUP_TO_SCENARIO = {
    **SCENARIO_LABELS,
    **{f"debiased_coarse/{g}": label for g, label in SCENARIO_LABELS.items()},
}


def group_leaves(group: str) -> dict[tuple[str, str, str], xr.DataArray]:
    """One lazy DataArray per (group, variable, member) under tree[group]."""
    node = tree[group]
    return {
        (group, v, m): node[f"{v}/{m}"].dataset[v]
        for v in VARIABLES
        if v in node.children
        for m in node[v].children
    }


# One lazy DataArray per (scenario, variable, member). Members stay separate rows: within a
# scenario they can span different time ranges (some CESM2 SSP245 members end 2070).
leaves = {}
for group in GROUP_TO_SCENARIO:
    leaves.update(group_leaves(group))


def table(rows: dict) -> pd.DataFrame:
    """One row per leaf, indexed (scenario, variable, member)."""
    return pd.DataFrame.from_dict(rows, orient="index").rename_axis(
        ["scenario", "variable", "member"]
    )


print(f"{len(leaves)} (scenario, variable, member) leaves to check")

26 (scenario, variable, member) leaves to check


In [5]:
# temp! trims out anti-meridian nan slice area
leaves = {k: da.sel(lon=slice(-180, 170)) for k, da in leaves.items()}

## Per-leaf stats

In [6]:
def leaf_stats(da: xr.DataArray) -> xr.Dataset:
    return xr.Dataset(
        {
            "n_nan_cells": da.isnull().sum(),
            "min": da.min(),
            "max": da.max(),
            "fingerprint": xr.concat(
                [da.min(SPATIAL), da.max(SPATIAL)],
                dim=pd.Index(["min", "max"], name="stat"),
            ),
        }
    )

## Check 1 -- no NaNs

Whole-array NaN cell count per leaf; pass = zero.

In [7]:
def check_no_nans(stats: dict) -> pd.DataFrame:
    nan_df = table({k: {"n_nan_cells": int(s.n_nan_cells)} for k, s in stats.items()})
    nan_df["pass"] = nan_df.n_nan_cells == 0
    return nan_df

## Check 2 -- no duplicate time slices


In [8]:
def duplicate_pairs(da: xr.DataArray, fp: xr.DataArray) -> list[tuple[str, str]]:
    """Confirmed identical-day pairs: fingerprint collision + exact elementwise equality."""
    arr = fp.transpose("time", "stat").values
    valid = ~np.isnan(arr).any(axis=1)
    times = fp.time.values[valid]

    _, inv = np.unique(arr[valid], axis=0, return_inverse=True)
    groups: dict[int, list[np.datetime64]] = {}
    for t, g in zip(times, inv):
        groups.setdefault(g, []).append(t)

    def fmt(t: np.datetime64) -> str:
        return str(np.datetime_as_string(t, unit="D"))

    return [
        (fmt(g[0]), fmt(t))
        for g in groups.values()
        if len(g) > 1
        for t in g[1:]
        # drop the scalar time coord -- .equals compares coords too, and the differing
        # timestamps would otherwise mask genuinely identical data
        if da.sel(time=t).drop_vars("time").equals(da.sel(time=g[0]).drop_vars("time"))
    ]


def check_no_duplicates(leaves: dict, stats: dict) -> pd.DataFrame:
    dup_df = table(
        {
            k: {"duplicate_pairs": duplicate_pairs(leaves[k], s.fingerprint)}
            for k, s in stats.items()
        }
    )
    dup_df["pass"] = dup_df.duplicate_pairs.str.len() == 0
    return dup_df

## Check 3 -- reasonable ranges


In [9]:
TEMP_VARS = {"tas", "tasmax", "tasmin"}
HOT_K = 65 + 273.15  # outlandishly_high_temp threshold
COLD_K = -100 + 273.15  # outlandishly_low_temp threshold


def irregular_days(v: str, fp: xr.DataArray) -> list[tuple[str, float]]:
    """(date, K) for days whose spatial max/min breaches the outlandish temp thresholds."""
    if v not in TEMP_VARS:
        return []
    smax = fp.sel(stat="max")
    smin = fp.sel(stat="min")
    bad = ((smax > HOT_K) | (smin < COLD_K)).values
    times = fp.time.values[bad]
    vals = smax.where(smax > HOT_K, smin).values[bad]  # report whichever extreme tripped
    return [(str(t)[:10], round(float(x), 1)) for t, x in zip(times, vals)]


def check_reasonable_ranges(stats: dict) -> pd.DataFrame:
    range_df = table(
        {
            k: {
                "min": float(s["min"]),
                "max": float(s["max"]),
                "irregular_days": irregular_days(k[1], s["fingerprint"]),
            }
            for k, s in stats.items()
        }
    )
    range_df["pass"] = [
        VAR_SPATIAL_RANGES[v]["min"][0] <= mn <= VAR_SPATIAL_RANGES[v]["min"][1]
        and VAR_SPATIAL_RANGES[v]["max"][0] <= mx <= VAR_SPATIAL_RANGES[v]["max"][1]
        for (s, v, m), mn, mx in zip(range_df.index, range_df["min"], range_df["max"])
    ]
    range_df["n_irregular_days"] = range_df.irregular_days.str.len()
    return range_df

## Check 4 -- within input time bounds


In [10]:
def check_time_bounds(leaves: dict) -> pd.DataFrame:
    bounds_df = table(
        {
            (s, v, m): {
                "actual_start": str(da.time.values[0])[:10],
                "actual_end": str(da.time.values[-1])[:10],
                "expected": resolve_member_time_bounds(GCM, GROUP_TO_SCENARIO[s], m),
            }
            for (s, v, m), da in leaves.items()
        }
    )
    expected_start = bounds_df.expected.str[0].fillna("0000-01-01")  # no known bounds -> pass
    expected_end = bounds_df.expected.str[1].fillna("9999-12-31")
    is_sai = bounds_df.index.get_level_values("scenario").str.upper().str.contains("G6|SAI")
    bounds_df["pass"] = (is_sai | (bounds_df.actual_start >= expected_start)) & (
        bounds_df.actual_end <= expected_end
    )
    return bounds_df

## Run checks and report

Single batched `dask.compute` for the lazy per-leaf stats, then run all four checks.

In [11]:
stats = dask.compute({key: leaf_stats(da) for key, da in leaves.items()})[0]
nan_df = check_no_nans(stats)
dup_df = check_no_duplicates(leaves, stats)
range_df = check_reasonable_ranges(stats)
bounds_df = check_time_bounds(leaves)

summary = pd.concat(
    {
        "no_nans": nan_df["pass"],
        "no_duplicate_timesteps": dup_df["pass"],
        "reasonable_range": range_df["pass"],
        "n_irregular_days": range_df["n_irregular_days"],
        "within_input_time_bounds": bounds_df["pass"],
    },
    axis=1,
)
print(summary.to_string())
check_cols = ["no_nans", "no_duplicate_timesteps", "reasonable_range", "within_input_time_bounds"]
print(f"\nAll checks passed across {len(summary)} leaves: {bool(summary[check_cols].all().all())}")

# Drill-down: dates + values for any leaf with implausible single-day spikes.
flagged = range_df[range_df.n_irregular_days > 0]
if len(flagged):
    print("\nIrregular days (report-only, spatial extreme past 65C / -100C):")
    for (s, v, m), row in flagged.iterrows():
        print(f"  {s}/{v}/{m}: {row.n_irregular_days} days")
        for date, k in row.irregular_days:
            print(f"    {date}  {k}K ({k - 273.15:.1f}C)")

                                              no_nans  no_duplicate_timesteps  reasonable_range  n_irregular_days  within_input_time_bounds
scenario                   variable member                                                                                                 
historical                 tas      r1i1p1f1     True                    True              True                 0                      True
                                    r2i1p1f1     True                    True              True                 0                      True
                                    r3i1p1f1     True                    True              True                 0                      True
                           tasmax   001          True                    True              True                 0                      True
ssp245                     tas      003          True                    True              True                 0                      True
                    

In [ ]:
client.shutdown()